# Module 7: Tracing using Langsmith

In [1]:
%pip install -qU langchain langchain-openai langchain-chroma langchain-text-splitters langsmith

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

c:\Users\vammun01\OneDrive - Robert Half\repos\prod-rag\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


## Setup Azure OpenAI environment variables & LLM and embedding models 
Copy content from previous module

In [3]:
# Initialize environment variables for Azure OpenAI
api_key = (os.getenv("AZURE_OPENAI_API_KEY") or "").strip()
embedding_deployment = (os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME") or "").strip()
endpoint = (os.getenv("AZURE_OPENAI_ENDPOINT") or "").strip()
llm_deployment = (
    os.getenv("AZURE_OPENAI_LLM_DEPLOYMENT_NAME")
    or os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")
    or os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
    or ""
).strip()


# Create embeddings client based on endpoint type
# Foundry project endpoint -> OpenAI-compatible v1 endpoint.
if "services.ai.azure.com" in endpoint and "/api/projects/" in endpoint:
    resource_root = endpoint.split("/api/projects/")[0]
    openai_endpoint = f"{resource_root}/openai/v1"
    embed_model = OpenAIEmbeddings(
        model=embedding_deployment,
        api_key=api_key,
        base_url=openai_endpoint,
    )
    embed_model.embed_query("endpoint-check")
    print("Embeddings client initialized OpenAI-compatible endpoint.")

    llm = ChatOpenAI(
        model=llm_deployment,
        api_key=api_key,
        base_url=openai_endpoint,
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    print("LLM initialized via OpenAI-compatible endpoint.")
else:
    api_version = (os.getenv("AZURE_OPENAI_API_VERSION") or "2024-02-01").strip()
    embed_model = AzureOpenAIEmbeddings(
        model=embedding_deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        openai_api_version=api_version,
    )
    embed_model.embed_query("endpoint-check")
    print(f"Embeddings client initialized via Azure OpenAI endpoint (api_version={api_version}).")

    llm = AzureChatOpenAI(
        azure_deployment=llm_deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version=api_version,
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    print(f"LLM initialized via Azure OpenAI endpoint (api_version={api_version}).")

Embeddings client initialized OpenAI-compatible endpoint.
LLM initialized via OpenAI-compatible endpoint.


In [4]:
from langchain_core.documents import Document
documents = [
    Document(
        page_content="LangChain helps build RAG systems using LLMs and retrievers.",
        metadata={"source": "langchain.pdf", "page": 1}
    ),
    Document(
        page_content="LangGraph enables multi-step workflows and agent orchestration.",
        metadata={"source": "langgraph.pdf", "page": 2}
    ),
    Document(
        page_content="LangSmith provides observability and evaluation for LLM systems.",
        metadata={"source": "langsmith.pdf", "page": 3}
    ),
]

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 3


In [6]:
from langchain_community.retrievers import BM25Retriever
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

In [7]:
from langchain_chroma import Chroma
# Embeddings and Persistent DB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embed_model,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [8]:
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata['source']} (Page {doc.metadata.get('page', 'N/A')})\n{doc.page_content}"
        for doc in docs
    )

In [9]:
def hybrid_retrieve(query):
    bm25_docs = bm25_retriever.invoke(query)
    vector_docs = retriever.invoke(query)
    
    # Combine and deduplicate
    combined = bm25_docs + vector_docs
    
    unique_docs = {doc.page_content: doc for doc in combined}
    
    return list(unique_docs.values())

## Re-rank - Simple

In [10]:
def simple_reranker(query, docs):
    def score(doc):
        return sum(word in doc.page_content.lower() for word in query.lower().split())
    
    ranked = sorted(docs, key=score, reverse=True)
    return ranked[:3]

## Re-Rank (Cross Encoder)

In [ ]:
%pip install -qU sentence-transformers truststore

Note: you may need to restart the kernel to use updated packages.


In [15]:
%pip install -qU transformers --use-feature=truststore

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install -qU truststore sentence-transformers huggingface_hub

import ssl
import truststore

# Patch ssl.create_default_context to use the Windows certificate store.
truststore.inject_into_ssl()

import httpx
from huggingface_hub import snapshot_download
from huggingface_hub.utils._http import close_session, set_client_factory

# Build an SSL context that trusts the OS/corporate proxy certs,
# then force the HF Hub httpx client to use it (httpx ignores
# truststore's monkey-patch because it defaults to certifi).
ssl_context = ssl.create_default_context()
close_session()
set_client_factory(lambda: httpx.Client(verify=ssl_context))

# Download the full model repo locally (uses patched SSL)
model_id = "cross-encoder/ms-marco-MiniLM-L-6-v2"
local_model_path = snapshot_download(repo_id=model_id)
print(f"Model downloaded to: {local_model_path}")

from sentence_transformers import CrossEncoder

# Load the CrossEncoder from the local snapshot
model = CrossEncoder(local_model_path)

# Rank a list of passages for a query
query = "How many people live in Berlin?"
passages = [
    "Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers.",
    "Berlin is well known for its museums.",
    "In 2014, the city state Berlin had 37,368 live births (+6.6%), a record number since 1991.",
    "The urban area of Berlin comprised about 4.1 million people in 2014, making it the seventh most populous urban area in the European Union.",
    "The city of Paris had a population of 2,165,423 people within its administrative city limits as of January 1, 2019",
    "An estimated 300,000-420,000 Muslims reside in Berlin, making up about 8-11 percent of the population.",
    "Berlin is subdivided into 12 boroughs or districts (Bezirke).",
    "In 2015, the total labour force in Berlin was 1.85 million.",
    "In 2013 around 600,000 Berliners were registered in one of the more than 2,300 sport and fitness clubs.",
    "Berlin has a yearly total of about 135 million day visitors, which puts it in third place among the most-visited city destinations in the European Union.",
]
ranks = model.rank(query, passages)

# Print the scores
print("Query:", query)
for rank in ranks:
    print(f"{rank['score']:.2f}\t{passages[rank['corpus_id']]}")

Note: you may need to restart the kernel to use updated packages.


c:\Users\vammun01\OneDrive - Robert Half\repos\prod-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Unlike embeddings:
 - query + doc are processed together
 - deeper semantic matching
 - better ranking

In [ ]:
def cross_encoder_rerank(query, docs, top_k=5):
    pairs = [(query, doc.page_content) for doc in docs]

    scores = model.predict(pairs)

    scored_docs = list(zip(docs, scores))

    ranked = sorted(scored_docs, key=lambda x: x[1], reverse=True)

    return [doc for doc, score in ranked[:top_k]]

## Problem Example

User Query:
"What does it do?"

This query is too vague for retrieval.

We need to transform it into something like:
"What does LangGraph do in LLM systems?"

In [11]:
def rewrite_query(query):
    rewrite_prompt = ChatPromptTemplate.from_template("""
    Rewrite the following query to be more specific and retrieval-friendly.

    Original Query:
    {query}

    Improved Query:
    """)
    
    response = llm.invoke(
        rewrite_prompt.invoke({"query": query})
    )
    
    return response.content.strip()

In [12]:
query = "What does it do?"
improved_query = rewrite_query(query)

print("Original:", query)
print("Rewritten:", improved_query)

Original: What does it do?
Rewritten: Improved Query:
What is the purpose and main function of **[the specific tool/feature/system]**, and what actions does it perform when used?


## Multi-Query Generation

Instead of one query, generate multiple variations.

In [13]:
def generate_multi_queries(query):
    multi_prompt = ChatPromptTemplate.from_template("""
    Generate 3 different variations of the query for better document retrieval.

    Query:
    {query}
    
    Variations:
    """)
    
    response = llm.invoke(
        multi_prompt.invoke({"query": query})
    )
    
    queries = response.content.split("\n")
    return [q.strip("- ").strip() for q in queries if q.strip()]

## Combine Multi-Query Retrieval

In [14]:
def multi_query_retrieve(query):
    queries = generate_multi_queries(query)
    
    all_docs = []
    for q in queries:
        docs = hybrid_retrieve(q)
        all_docs.extend(docs)
    
    # Deduplicate
    unique_docs = {doc.page_content: doc for doc in all_docs}
    
    return list(unique_docs.values())

## Rerank the Combined Results

In [15]:
def full_retrieval_pipeline(query):
    rewritten = rewrite_query(query)
    docs = multi_query_retrieve(rewritten)
    reranked = simple_reranker(rewritten, docs)
    
    return reranked

In [16]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.

Include source citations in your answer.

Context:
{context}

Question:
{question}
""")

In [17]:
def ask_advanced_rag(query):
    docs = full_retrieval_pipeline(query)
    
    context = format_docs(docs)
    
    final_prompt = prompt.invoke({
        "context": context,
        "question": query
    })
    
    response = llm.invoke(final_prompt)
    
    return {
        "answer": response.content,
        "docs": docs
    }

## Step 1: Create Evaluation Dataset

Each test case includes:
- question
- expected answer
- expected keywords

In [18]:
eval_dataset = [
    {
        "question": "What does LangGraph do?",
        "ground_truth": "LangGraph is used for orchestrating workflows and agents",
    },
    {
        "question": "What is LangSmith used for?",
        "ground_truth": "LangSmith helps with debugging, evaluation, and observability of LLM applications",
    }
]

## Step 2: Evaluation Prompt (LLM-as-a-Judge)

This is what RAGAS uses internally.

In [19]:
def judge_answer(question, answer, ground_truth):
    eval_prompt = f"""
    You are an evaluator.

    Question: {question}
    Ground Truth: {ground_truth}
    Model Answer: {answer}

    Score the answer from 0 to 1 based on correctness and completeness.

    Only return a number.
    """
    
    response = llm.invoke(eval_prompt)
    return float(response.content.strip())

## Step 3: Check Faithfullness

In [20]:
# Ground Checker for Faithfulness
def check_faithfulness(answer, context):
    prompt = f"""
    Check if the answer is supported by the context.

    Context:
    {context}

    Answer:
    {answer}

    Return YES or NO.
    """
    
    response = llm.invoke(prompt)
    return response.content.strip()

## Step 4: Retrieval Quality Check

In [21]:
def check_retrieval(question, docs):
    context = "\n\n".join([d.page_content for d in docs])
    
    prompt = f"""
    Question: {question}

    Context:
    {context}

    Does the context contain enough information to answer the question?

    Answer YES or NO.
    """
    
    response = llm.invoke(prompt)
    return response.content.strip()

## Step 5: Full Evaluation Pipeline

In [22]:
def evaluate_rag_system(dataset):
    results = []
    
    for item in dataset:
        rag_result = ask_advanced_rag(item["question"])
        
        answer = rag_result["answer"]
        docs = rag_result["docs"]
        context = format_docs(docs)
        
        correctness = judge_answer(
            item["question"], answer, item["ground_truth"]
        )
        
        faithfulness = check_faithfulness(answer, context)
        retrieval = check_retrieval(item["question"], docs)
        
        results.append({
            "question": item["question"],
            "correctness": correctness,
            "faithfulness": faithfulness,
            "retrieval": retrieval
        })
    
    return results

In [23]:
def classify_failure(result):
    if result["retrieval"] == "NO":
        return "Retrieval Failure"
    elif result["faithfulness"] == "NO":
        return "Hallucination / Grounding Failure"
    elif result["correctness"] < 0.5:
        return "Generation Failure"
    else:
        return "Pass"

In [24]:
results = evaluate_rag_system(eval_dataset)

for r in results:
    failure_type = classify_failure(r)
    
    print("\nQuestion:", r["question"])
    print("Correctness:", r["correctness"])
    print("Faithfulness:", r["faithfulness"])
    print("Retrieval:", r["retrieval"])
    print("Failure Type:", failure_type)


Question: What does LangGraph do?
Correctness: 1.0
Faithfulness: YES
Retrieval: YES
Failure Type: Pass

Question: What is LangSmith used for?
Correctness: 0.9
Faithfulness: YES
Retrieval: YES
Failure Type: Pass


## Guardrail 1: Strict Prompt Enforcement

In [25]:
strict_prompt = ChatPromptTemplate.from_template("""
You are a strict assistant.

Rules:
- Answer ONLY using the provided context
- Do NOT use prior knowledge
- If the answer is not in the context, say:
  "I do not have enough information to answer this question."

Context:
{context}

Question:
{question}
""")

## Guardrail 2: Faithfulness Check (Post-Validation)

In [26]:
def is_faithful(answer, context):
    prompt = f"""
    Determine if the answer is fully supported by the context.

    Context:
    {context}

    Answer:
    {answer}

    Return YES or NO.
    """
    
    response = llm.invoke(prompt)
    return response.content.strip()

## Guardrail 3: Confidence Scoring

In [27]:
def confidence_score(answer, context):
    prompt = f"""
    Score the confidence of this answer based on context support.

    Context:
    {context}

    Answer:
    {answer}

    Return a score from 0 to 1.
    """
    
    response = llm.invoke(prompt)
    return float(response.content.strip())

In [28]:
def safe_rag(query):
    docs = full_retrieval_pipeline(query)
    context = format_docs(docs)
    
    # Generate answer
    final_prompt = strict_prompt.invoke({
        "context": context,
        "question": query
    })
    
    response = llm.invoke(final_prompt)
    answer = response.content
    
    # Guardrails
    faithfulness = is_faithful(answer, context)
    confidence = confidence_score(answer, context)
    
    if faithfulness == "NO" or confidence < 0.6:
        return {
            "answer": "I do not have enough reliable information to answer this question.",
            "status": "rejected",
            "confidence": confidence
        }
    
    return {
        "answer": answer,
        "status": "accepted",
        "confidence": confidence
    }

In [29]:
r = safe_rag("What does LangGraph do?")
print("Answer:\n", r["answer"])
print("\nStatus:", r["status"])
print("\nConfidence:", r["confidence"])

Answer:
 LangGraph enables multi-step workflows and agent orchestration.

Status: accepted

Confidence: 1.0


In [30]:
r = safe_rag("What is Azure?")
print("Answer:\n", r["answer"])
print("\nStatus:", r["status"])
print("\nConfidence:", r["confidence"])

Answer:
 I do not have enough information to answer this question.

Status: accepted

Confidence: 1.0


## Langsmith: Traceability
https://docs.langchain.com/langsmith/create-account-api-key

- LANGSMITH_TRACING=true
- LANGSMITH_ENDPOINT=https://api.smith.langchain.com
- LANGSMITH_API_KEY=your-api-key
- LANGSMITH_PROJECT=project-name

Please note: Restart the kernel if the dashboard is not showing up the traces.

In [31]:
from langsmith import traceable

@traceable
def traced_safe_rag(query):
    docs = full_retrieval_pipeline(query)
    context = format_docs(docs)
    
    final_prompt = strict_prompt.invoke({
        "context": context,
        "question": query
    })
    
    response = llm.invoke(final_prompt)
    answer = response.content
    
    # Guardrails
    faithfulness = is_faithful(answer, context)
    confidence = confidence_score(answer, context)
    
    return {
        "query": query,
        "answer": answer,
        "faithfulness": faithfulness,
        "confidence": confidence,
        "docs": docs
    }

In [32]:
@traceable
def evaluated_run(question):
    result = traced_safe_rag(question)
    
    eval_score = judge_answer(
        question,
        result["answer"],
        "ground_truth_placeholder"
    )
    
    return {
        **result,
        "evaluation_score": eval_score
    }

In [33]:
result = evaluated_run("What does LangGraph do?")
print(result["answer"])
print(result["evaluation_score"])

LangGraph enables multi-step workflows and agent orchestration.
0.7
